# QuantumXCT — figure regeneration & customization

One notebook that regenerates **every** review figure for **both** datasets
(synthetic and ovarian), in **PNG (300 dpi), PDF and SVG**.

All figures are produced by the shared tools in `analysis_tools/`, which loop over
both datasets in the *same* code, so the synthetic and ovarian figure sets are
identical by construction — they cannot drift apart.

Run the bulk cells to regenerate everything, or jump to the **Customize** section
to tweak one figure (colors, size, title, fonts) and re-save it.

Outputs:
- `results/analyses/figures/`  — data histograms, angle-init curves
- `figures/search_panels/{dataset}_{l1,l2}/` — circuits, networks, occupancy, angle-opt traces


## 0. Setup — run from `QuantumXCT/python`

In [ ]:
import os, sys
# make sure we are in QuantumXCT/python (edit if your path differs)
if os.path.basename(os.getcwd()) != "python":
    os.chdir(os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd())=="analysis_tools" else os.getcwd())
sys.path.insert(0, os.getcwd())
print("cwd:", os.getcwd())
import matplotlib.pyplot as plt
%matplotlib inline

## 1. Data joint histograms  (Frequency, bit strings, value labels)
Legacy style: y = raw counts, x = `Bit String (qN ... q0)`, a value label on every bar.
Titles `Data Joint Histogram {Non-Interacting|Interacting}-CT{1,2}`.

In [ ]:
import importlib, analysis_tools.make_data_histograms as mdh
importlib.reload(mdh)
mdh.main()   # writes *_data_hist_* .png/.pdf/.svg for synthetic and ovarian

## 2. Angle-initialization study (0 vs π/2) — both datasets

In [ ]:
import analysis_tools.run_angle_init as rai
importlib.reload(rai)
rai.main()   # writes *_angle_init_curves .png/.pdf/.svg  and angle_init_study.csv

## 3. Search-panel figures (circuits, networks, occupancy, angle-opt)
N-wise / Multi-Epoch / VQE+Multi-Epoch  x  GRN / LR / GRN+LR, per dataset & encoding.
This is the slow one (~30–40 min for all 4 dataset×encoding combos).
Restrict with `--datasets` / `--norms` while iterating.

In [ ]:
import subprocess, sys
# all four:  python analysis_tools/make_search_figures.py
# just what you need for the poster (fast):
subprocess.run([sys.executable, "analysis_tools/make_search_figures.py",
                "--datasets", "ovarian", "synthetic", "--norms", "l1"])

## 4. Customize a single figure

Everything above uses helper functions you can import and re-plot inline, then
save in all three formats. Example: redraw one **data histogram** with your own
colors / size / title, and save PNG+PDF+SVG to a `custom/` folder.
Edit the variables in the next cell.

In [ ]:
import os
import matplotlib.pyplot as plt
from analysis_tools.make_data_histograms import counts_vec
from analysis_tools.data import load_sim, load_ovarian, SIM_CT1, SIM_CT2, OV_CT1, OV_CT2

# --- edit these ---------------------------------------------------------
dataset   = "synthetic"          # "synthetic" or "ovarian"
condition = "ct1_co"             # ct1_mo / ct2_mo / ct1_co / ct2_co
title     = "Data Joint Histogram Interacting-CT1"
bar_color = "steelblue"          # any matplotlib color / hex
figsize   = (5.5, 4.2)
label_fs  = 9
out_name  = "custom/my_hist"     # saved under results/analyses/figures/
# ------------------------------------------------------------------------

H = (load_sim() if dataset=="synthetic" else load_ovarian())
genes = (SIM_CT1+SIM_CT2 if dataset=="synthetic" else OV_CT1+OV_CT2)
n = len(SIM_CT1 if dataset=="synthetic" and "ct1" in condition else
        SIM_CT2 if dataset=="synthetic" else
        OV_CT1 if "ct1" in condition else OV_CT2)
y = counts_vec(H[condition], n); x = list(range(2**n))
fig, ax = plt.subplots(figsize=figsize)
ax.bar(x, y, color=bar_color)
ax.set_xticks(x); ax.set_xticklabels([format(i, f"0{n}b") for i in x], rotation=90, fontsize=label_fs)
ax.set_xlabel(f"Bit String (q{n-1} ... q0)"); ax.set_ylabel("Frequency"); ax.set_title(title)
for xi, yi in zip(x, y):
    ax.annotate(str(yi), (xi, yi), textcoords="offset points", xytext=(0,2), ha="center", fontsize=7)
fig.tight_layout()
base = os.path.join("results/analyses/figures", out_name); os.makedirs(os.path.dirname(base), exist_ok=True)
for ext, kw in [("png", dict(dpi=300)), ("pdf", {}), ("svg", {})]:
    fig.savefig(base+"."+ext, bbox_inches="tight", **kw)
print("saved", base+".{png,pdf,svg}")
plt.show()

## 5. Customize a search panel (occupancy / circuit / network)

`make_search_figures` exposes `run_one(...)` (runs the search + exact re-optimization)
and the drawing helpers `plot_occupancy`, `draw_network`, `draw_circuit`, `plot_trace`.
Example: re-plot the ovarian GRN+LR multi-epoch **occupancy** with custom colors.

In [ ]:
import importlib, analysis_tools.make_search_figures as msf
importlib.reload(msf)
from qxct.exact import marginals

D = msf.DATASETS["ovarian"]; H = D["loader"](); genes = D["ct1"]+D["ct2"]; n1=len(D["ct1"]); n2=len(D["ct2"])
r = msf.run_one(H, genes, norm="l1", scope="all", algo="multi_epoch", kl_tol=D["kl_tol"], vqe_k=16)
sc = r["sc"]; pm1, pm2 = marginals(sc.state(r["topo"], r["angles"]), sc.n1, sc.n2)
p01, p02 = marginals(sc.psi0, sc.n1, sc.n2)

# tweak the module colors before drawing (optional):
# msf.C_NON, msf.C_QX, msf.C_INT = "steelblue", "mediumpurple", "indigo"
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4.5), sharey=True, gridspec_kw={"width_ratios":[1,1.25]})
msf.plot_occupancy(a1, p01, pm1, sc.q1, n1, "CT1")
msf.plot_occupancy(a2, p02, pm2, sc.q2, n2, "CT2", ylabel=False)
fig.tight_layout()
for ext, kw in [("png",dict(dpi=300)),("pdf",{}),("svg",{})]:
    fig.savefig(f"results/analyses/figures/custom/ovarian_occupancy_custom.{ext}", bbox_inches="tight", **kw)
plt.show()